# Battery Cathode ML - 使用教程

本教程介绍如何使用电池正极材料机器学习预测项目。

## 目录
1. [环境设置](#环境设置)
2. [快速开始](#快速开始)
3. [数据准备](#数据准备)
4. [模型训练](#模型训练)
5. [新材料预测](#新材料预测)
6. [候选材料筛选](#候选材料筛选)
7. [高级用法](#高级用法)

## 环境设置

In [ ]:
# 安装依赖（如果还未安装）
!pip install -r requirements.txt

In [ ]:
# 导入必要的库
import sys
from pathlib import Path

# 添加 src 到路径
PROJECT_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src").is_dir() and (p / "README.md").is_file())
src_path = PROJECT_ROOT / "src"
sys.path.insert(0, str(src_path))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

print("Environment setup complete!")

## 快速开始

### 加载训练好的模型

In [ ]:
from predictor import load_models

# 加载所有训练好的模型
models = load_models()
print(f"Loaded models: {list(models.keys())}")

### 预测单个材料的性质

In [ ]:
from predictor import predict_single

# 预测 LiFePO4 的性质
formula = "LiFePO4"
result = predict_single(formula, models)

print(f"\nMaterial: {result['formula']}")
if 'Egap_ensemble_eV' in result:
    print(f"Band Gap: {result['Egap_ensemble_eV']:.3f} eV ({result.get('conductivity', 'N/A')})")
if 'Eform_ensemble_eV_fu' in result:
    print(f"Formation Energy: {result['Eform_ensemble_eV_fu']:.3f} eV/fu")
if 'Voltage_ensemble_V' in result:
    print(f"Average Voltage: {result['Voltage_ensemble_V']:.3f} V ({result.get('voltage_level', 'N/A')})")

## 数据准备

### 生成模拟电压数据

In [ ]:
from generate_voltage_data import generate_voltage_data

# 生成模拟数据
df_voltage = generate_voltage_data(n_samples=500)

print(f"Generated {len(df_voltage)} samples")
print("\nVoltage distribution by family:")
print(df_voltage.groupby('family')['average_voltage_V'].agg(['mean', 'std', 'count']).round(3))

In [ ]:
# 可视化电压分布
plt.figure(figsize=(10, 6))
df_voltage.boxplot(column='average_voltage_V', by='family')
plt.suptitle('Voltage Distribution by Material Family')
plt.title('')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

### 特征提取示例

In [ ]:
from features import featurize_one, build_universe
from formula import parse_formula

# 构建元素宇宙
test_formulas = ['LiFePO4', 'LiCoO2', 'LiMn2O4']
universe = build_universe(test_formulas)
print(f"Element universe: {universe}")

# 提取特征
formula = "LiCoO2"
features = featurize_one(formula, element_list=universe)

print(f"\nFeatures for {formula}:")
for key, value in list(features.items())[:10]:
    print(f"  {key}: {value:.4f}")
print(f"  ... ({len(features)} total features)")

## 模型训练

### 训练电压预测模型（演示）

In [ ]:
# 注意：实际训练中，建议使用更大的数据集和完整的训练脚本
# 这里演示如何手动训练一个简单的模型

from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, r2_score

# 使用模拟数据
X_features = []
y_voltage = []

for _, row in df_voltage.head(100).iterrows():
    feats = featurize_one(row['formula'], element_list=universe)
    if feats is not None:
        X_features.append(list(feats.values()))
        y_voltage.append(row['average_voltage_V'])

X_df = pd.DataFrame(X_features)
y = np.array(y_voltage)

# 分割训练集和测试集
X_train, X_test, y_train, y_test = train_test_split(
    X_df, y, test_size=0.2, random_state=42
)

# 训练模型
model = RandomForestRegressor(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

# 评估
y_pred = model.predict(X_test)
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print(f"Model Performance:")
print(f"  MAE: {mae:.4f} V")
print(f"  R²: {r2:.4f}")

## 新材料预测

### 批量预测多个材料

In [ ]:
from predictor import predict_material_properties

# 定义要预测的材料列表
candidate_formulas = [
    "LiFePO4",           # 橄榄石
    "LiCoO2",            # 层状氧化物
    "LiMn2O4",           # 尖晶石
    "LiNi0.8Co0.1Mn0.1O2",  # NMC811
    "Li3V2(PO4)3",       # 聚阴离子
    "Li2MnSiO4",         # 硅酸盐
]

# 批量预测
results = predict_material_properties(candidate_formulas)

# 显示结果
display(results)

### 自定义预测函数

In [ ]:
def analyze_material(formula, models):
    """综合分析一个材料的性能指标。"""
    result = predict_single(formula, models)
    
    print(f"\n{'='*60}")
    print(f"材料分析：{formula}")
    print(f"{'='*60}")
    
    # 基本信息
    if 'parsed' in result:
        print(f"化学式组成：{result['parsed']}")
    if 'molar_mass_g_mol' in result:
        print(f"摩尔质量：{result['molar_mass_g_mol']:.2f} g/mol")
    
    # 带隙分析
    if 'Egap_ensemble_eV' in result:
        gap = result['Egap_ensemble_eV']
        conductivity = result.get('conductivity', 'Unknown')
        print(f"\n电学性能:")
        print(f"  带隙：{gap:.3f} eV")
        print(f"  导电性分类：{conductivity}")
        
        # 建议
        if gap > 3.0:
            print(f"  💡 建议：宽带隙材料，可能需要掺杂改善导电性")
        elif gap < 0.5:
            print(f"  ⚠️ 注意：窄带隙可能导致电子漏电")
    
    # 电压分析
    if 'Voltage_ensemble_V' in result:
        voltage = result['Voltage_ensemble_V']
        voltage_level = result.get('voltage_level', 'Unknown')
        print(f"\n电化学性能:")
        print(f"  平均电压：{voltage:.3f} V")
        print(f"  电压等级：{voltage_level}")
        
        # 建议
        if voltage > 4.5:
            print(f"  ⚠️ 注意：高电压可能导致电解液分解")
        elif voltage < 3.0:
            print(f"  💡 考虑：低电压材料能量密度较低")
        else:
            print(f"  ✅ 理想的电压范围")
    
    print(f"{'='*60}\n")
    
    return result

# 分析几个典型材料
materials_to_analyze = ["LiFePO4", "LiCoO2", "LiNi0.5Mn1.5O4"]
for material in materials_to_analyze:
    analyze_material(material, models)

## 候选材料筛选

### 从候选列表中筛选最佳材料

In [ ]:
from predictor import screen_candidates

# 筛选电压在 3.0-5.0V 之间的 Top 10 材料
top_candidates = screen_candidates(
    n_top=10,
    min_voltage=3.0,
    max_voltage=5.0
)

### 可视化筛选结果

In [ ]:
import matplotlib.pyplot as plt

# 读取筛选结果
ranking_file = PROJECT_ROOT / "results" / "cathode_ranking.csv"
if ranking_file.exists():
    ranking_df = pd.read_csv(ranking_file, encoding='utf-8-sig')
    
    # Top 15 可视化
    top15 = ranking_df.head(15)
    
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    
    # 综合评分
    axes[0].barh(range(len(top15)), top15['score_total'], color='steelblue')
    axes[0].set_xlabel('综合评分')
    axes[0].set_title('Top 15 候选材料 - 综合评分')
    axes[0].invert_yaxis()
    
    # 电压
    if 'Voltage_ensemble_V' in top15.columns:
        axes[1].barh(range(len(top15)), top15['Voltage_ensemble_V'], color='coral')
        axes[1].set_xlabel('平均电压 (V)')
        axes[1].set_title('平均电压分布')
        axes[1].invert_yaxis()
    
    # 稳定性
    if 'Eform_ensemble_eV_fu' in top15.columns:
        axes[2].barh(range(len(top15)), top15['Eform_ensemble_eV_fu'], color='green')
        axes[2].set_xlabel('形成能 (eV/fu)')
        axes[2].set_title('形成能（越低越稳定）')
        axes[2].invert_yaxis()
    
    plt.tight_layout()
    plt.show()
else:
    print("未找到排名文件，请先运行 screening.py")

## 高级用法

### 自定义材料设计

In [ ]:
def design_better_cathode(base_formula, substitution_rules):
    """
    基于替换规则设计新的正极材料。
    
    Args:
        base_formula: 基础化学式
        substitution_rules: 替换规则，如 {'Fe': 'Mn'} 表示将 Fe 替换为 Mn
    """
    from formula import parse_formula, canonical_formula
    
    # 解析基础化学式
    composition = parse_formula(base_formula)
    
    # 应用替换规则
    new_composition = composition.copy()
    for old_element, new_element in substitution_rules.items():
        if old_element in new_composition:
            new_composition[new_element] = new_composition.pop(old_element)
    
    # 生成新化学式
    new_formula = canonical_formula(new_composition)
    
    return new_formula

# 示例：从 LiFePO4 出发，尝试不同的金属替换
base = "LiFePO4"
variations = [
    {"Fe": "Mn"},      # LiMnPO4
    {"Fe": "Co"},      # LiCoPO4
    {"Fe": "Ni"},      # LiNiPO4
    {"Fe": "Mg"},      # LiMgPO4
]

print(f"Base material: {base}")
print("\nProposed variations:")
for rule in variations:
    new_formula = design_better_cathode(base, rule)
    result = predict_single(new_formula, models)
    voltage = result.get('Voltage_ensemble_V', 'N/A')
    gap = result.get('Egap_ensemble_eV', 'N/A')
    print(f"  {new_formula}: Voltage={voltage}, BandGap={gap}")

### 不确定性量化

In [ ]:
def quantify_prediction_uncertainty(formula, models):
    """
    量化预测的不确定性（使用双模型离散度作为代理）。
    
    Returns:
        dict: 包含预测值和不确定性估计
    """
    from predictor import predict_single
    
    result = predict_single(formula, models)
    
    uncertainty_info = {
        'formula': formula,
    }
    
    # 带隙不确定性
    if 'Egap_spread' in result:
        spread = result['Egap_spread']
        uncertainty_info['gap_uncertainty'] = spread
        if spread > 0.2:
            uncertainty_info['gap_reliability'] = 'Low'
        elif spread > 0.1:
            uncertainty_info['gap_reliability'] = 'Medium'
        else:
            uncertainty_info['gap_reliability'] = 'High'
    
    return uncertainty_info

# 测试几个材料的预测不确定性
test_materials = ["LiFePO4", "LiCoO2", "Li2FeSiO4"]

print("Prediction Uncertainty Analysis:\n")
for material in test_materials:
    info = quantify_prediction_uncertainty(material, models)
    print(f"Material: {material}")
    if 'gap_uncertainty' in info:
        print(f"  Band Gap Spread: {info['gap_uncertainty']:.4f} eV")
        print(f"  Reliability: {info['gap_reliability']}")
    print()

## 总结

通过本教程，您已经学习了：

✅ **基础功能**：
- 加载和使用预训练模型
- 预测单个或批量材料的性质
- 特征工程和数据分析

✅ **进阶功能**：
- 材料筛选和排序
- 可视化分析
- 自定义材料设计
- 不确定性量化

### 下一步

- 查看 [README.md](../README.md) 了解更多详细信息
- 阅读 [API 文档](https://github.com/YOUR_USERNAME/battery-cathode-ml/wiki/API-Documentation)
- 尝试修改参数，探索不同材料组合
- 贡献代码，帮助改进这个项目！

---

**祝您好运，祝您研究顺利！** 🎉